# Kapitel 1 — Daten lesen · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch01_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die Zellen, die Sie bearbeiten sollen, sind mit **ZU ERGÄNZEN** markiert. Führen Sie die Zellen der Reihe nach von oben nach unten aus: Solange eine Antwort fehlt, melden die Prüfzellen dies, ohne einen Fehler auszulösen. Die Lösung finden Sie in `ch01_corrige.ipynb` auf derselben Begleitseite.

**Konvention.** Wie im Buch teilen Varianz, Standardabweichung, z-Wert und Kovarianz durch $n$ (`ddof=0`), sofern nicht ausdrücklich anders angegeben.

**Inhalt.** Daten · Die Berechnungen der Lektionen · Aufgaben 1.1 bis 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Farben des Buchs: eine Farbe = eine Rolle (beobachtet, vorhergesagt, Abweichung, Gerade des Kollegen)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Daten

Die Kennungen (D0.1, D1.1…) sind die des Buchs und der CSV-Dateien.

In [ ]:
# Daten von Kapitel 1, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D0.1 — die sechs Wohnungen der Einleitung
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — Verkaufspreise der T2-Wohnungen (zwei Zimmer, k€), Lektion 1.1
north = [174, 177, 181, 183, 185]
south  = [150, 165, 185, 195, 205]

# D1.2 — Verkaufsregister des Viertels Süd, Lektionen 1.2 bis 1.5
sales_log = pd.DataFrame({
    'sale':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'rooms':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'floor_area':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'energy_class':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'price':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'days_to_sell': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — Preis pro m² nach Stockwerk (k€/m²), Lektion 1.5
floors = pd.DataFrame({
    'floor':        [0, 1, 2, 3, 4, 5, 6],
    'price_per_m2': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — Verkaufsdauern von drei Agenturen (Tage), Aufgabe 1.1
agency_delays = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — Monatsmieten der Studios (€), Aufgabe 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `home` | Kennung der Wohnung (A, B…) | — | `logement` |
| `floor_area` | Wohnfläche der Wohnung | m² | `surface_m2` |
| `price` | Verkaufspreis | k€ | `prix_kEUR` |
| `sale` | Kennung des Verkaufs (S01…) | — | `vente` |
| `rooms` | Anzahl der Zimmer | Zimmer | `pieces` |
| `energy_class` | Energieeffizienzklasse (A bis G) | — | `dpe` |
| `days_to_sell` | Verkaufsdauer | Tage | `delai_jours` |
| `floor` | Stockwerk | — | `etage` |
| `price_per_m2` | Preis pro Quadratmeter | k€/m² | `prix_m2_kEUR` |

In [ ]:
# Bereitgestellte Werkzeuge: Sie müssen sie nicht ändern

def book_quartiles(values):
    """Q1 und Q3 nach der Regel des Buchs (Gleichung 1.5): Ränge ⌈n/4⌉ und ⌈3n/4⌉."""
    x = sorted(values)
    n = len(x)
    rank_q1 = -(-n // 4)          # n/4 auf die nächste ganze Zahl aufgerundet
    rank_q3 = -(-3 * n // 4)      # 3n/4 auf die nächste ganze Zahl aufgerundet
    return x[rank_q1 - 1], x[rank_q3 - 1]   # Rang 1 = Position 0 in Python


def dot_rows(series, xlabel, bands=None, ax=None):
    """Eine Punktreihe pro Datenreihe, wie in den Abbildungen von Lektion 1.1.
    bands: Wörterbuch {Name: (Mittelwert, Standardabweichung)}, um Mittelwert ± Standardabweichung zu zeichnen."""
    if ax is None:
        fig, ax = plt.subplots()
    names = list(series)
    colors = [BLUE, ORANGE, '#52514e']
    for i, name in enumerate(names):
        y = len(names) - 1 - i
        if bands and name in bands:
            m, s = bands[name]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[name], [y] * len(series[name]), s=50,
                   color=colors[i % 3], zorder=3)
    ax.set_yticks(range(len(names)), list(reversed(names)))
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Die Berechnungen der Lektionen, in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 1.1 bis 1.5. Führen Sie sie aus und vergleichen Sie mit dem Buch. Hier werden `np.std` und `np.var` mit `ddof=0` aufgerufen, also mit einer Division durch $n$, wie im Kapitel.

In [ ]:
# Lektion 1.1 — Mittelwert, Varianz und Standardabweichung (Division durch n)
for name, one_series in [('Nord', north), ('Süd', south)]:
    print(f'{name}: Mittelwert = {np.mean(one_series):g} k€, Varianz = {np.var(one_series, ddof=0):g} k€², '
          f'Standardabweichung = {np.std(one_series, ddof=0):g} k€')

In [ ]:
# Lektion 1.2 — Häufigkeiten je Energieeffizienzklasse (ordinale qualitative Variable)
sales_log['energy_class'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lektion 1.3 — Mittelwert, Median und Quartile der Verkaufsdauern
d = sales_log['days_to_sell']
print('Mittelwert:', d.mean(), ' Median:', d.median())
print('ohne S13: Mittelwert', d[sales_log['sale'] != 'S13'].mean(),
      ' Median', d[sales_log['sale'] != 'S13'].median())
print('Quartile, Regel des Buchs (Gleichung 1.5):', book_quartiles(d))
print('Quartile, NumPy-Standard (Interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lektion 1.4 — standardisierte Werte (Gleichung 1.6, Standardabweichung mit Division durch n)
for name, one_series, sale in [('Nord', north, 188), ('Süd', south, 210)]:
    m, s = np.mean(one_series), np.std(one_series, ddof=0)
    print(f'{name}: z der Reihe = {np.round((np.array(one_series) - m) / s, 4)}, '
          f'z des Verkaufs zu {sale} k€ = {(sale - m) / s:g}')

In [ ]:
# Lektion 1.5 — Kovarianz und Korrelation der fünf T2-Wohnungen im Süden (Division durch n)
t2 = sales_log[sales_log['rooms'] == 2]
x, y = t2['floor_area'].to_numpy(float), t2['price'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('Kovarianz:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r mit np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r Stockwerk / Preis pro m²:', np.corrcoef(floors['floor'], floors['price_per_m2'])[0, 1].round(4))

## Aufgabe 1.1 — Ein Diagramm lesen

*Daten D1.E1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1).* Das folgende Diagramm gibt das aus Aufgabe 1.1 des Buchs wieder.

In [ ]:
dot_rows({f'Agentur {a}': v for a, v in agency_delays.items()},
                  'Verkaufsdauer (Tage)')
plt.xlim(0, 60)
plt.show()

**a) und b)** Schätzen Sie nach Augenmaß die mittlere Verkaufsdauer jeder Agentur und ordnen Sie die Agenturen dann von der am wenigsten zur am stärksten gestreuten.

In [ ]:
# ZU ERGÄNZEN: Ihre Schätzungen nach Augenmaß, vor jeder Rechnung
estimated_mean = {'P': None, 'Q': None, 'R': None}   # in Tagen
spread_order = None   # Liste, von der am wenigsten zur am stärksten gestreuten, z. B. ['Q', 'P', 'R']

In [ ]:
# Prüfung (Division durch n)
for a, v in agency_delays.items():
    print(f'Agentur {a}: Mittelwert = {np.mean(v):g} T, Standardabweichung = {np.std(v, ddof=0):.2f} T')
true_order = sorted(agency_delays, key=lambda a: np.std(agency_delays[a]))
if spread_order is None or None in estimated_mean.values():
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Reihenfolge der Streuung:', 'richtig' if spread_order == true_order else 'bitte prüfen')

**c)** Zählen Sie für Q und für R die Verkäufe, die höchstens 2 Tage vom Mittelwert entfernt abgeschlossen wurden.

In [ ]:
# ZU ERGÄNZEN: Ihre Zählungen, aus dem Diagramm abgelesen
close_to_Q = None
close_to_R = None

In [ ]:
# Prüfung
for a, yours in [('Q', close_to_Q), ('R', close_to_R)]:
    v = np.array(agency_delays[a])
    truth = int(np.sum(np.abs(v - v.mean()) <= 2))
    if yours is None:
        print(f'Agentur {a}: noch nicht ausgefüllt.')
    else:
        print(f"Agentur {a}: {'richtig' if yours == truth else 'bitte prüfen'}")

*Ihre Erklärung zu c):* …

## Aufgabe 1.2 — Zwei Wohnanlagen, dieselbe mittlere Miete

*Daten D1.E2. Begriffe: Mittelwert, Varianz und Standardabweichung (Lektion 1.1).* Camille kann höchstens 570 € im Monat ausgeben. Die Fragen a) bis d) lösen Sie zuerst auf Papier, mit der Tabelle aus Aufgabe 1.2 des Buchs. Dieses Notebook dient dazu, Ihre Antworten zu prüfen, dann die Standardabweichung zu programmieren (Frage e) und zwei Konventionen zu vergleichen (Frage f).

In [ ]:
dot_rows({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monatsmiete eines Studios (€)")
plt.axvline(570, color='black', linestyle=':', label='Budget von Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Die Mieten ablesen

Wie viele Studios kann Camille in jeder Wohnanlage mit 570 € mieten, und wie viele mit 620 €?

In [ ]:
# ZU ERGÄNZEN: Anzahl der bezahlbaren Studios, aus der Mietliste abgelesen
affordable = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Prüfung: `<=` liefert Wahr/Falsch-Werte, np.sum zählt die „Wahr“-Werte
for budget, answers in affordable.items():
    for name, one_series in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        truth = int(np.sum(np.array(one_series) <= budget))
        yours = answers[name]
        status = 'noch nicht ausgefüllt' if yours is None else ('richtig' if yours == truth else 'bitte prüfen')
        print(f'Budget {budget} €, {name}: {status}')

*Ihre Antwort zu a), in einem Satz:* …

### b) Die beiden Mittelwerte

Berechnen Sie sie auf Papier und tragen Sie Ihre Ergebnisse dann unten ein.

In [ ]:
# ZU ERGÄNZEN: Ihre von Hand berechneten Mittelwerte (in €)
mean_tilleuls_by_hand = None
mean_port_by_hand = None

In [ ]:
# Prüfung
for name, one_series, yours in [('Les Tilleuls', tilleuls, mean_tilleuls_by_hand),
                              ('Le Port', port, mean_port_by_hand)]:
    if yours is None:
        print(f'{name}: bitte füllen Sie die vorherige Zelle aus.')
    else:
        ok = np.isclose(yours, np.mean(one_series))
        print(f"{name}: {'richtig' if ok else 'bitte prüfen'} (Ihr Wert: {yours} €)")

### c) Varianz und Standardabweichung von Hand (Division durch $n = 6$)

Erstellen Sie auf Papier für jede Wohnanlage die Tabelle der Abweichungen vom Mittelwert und ihrer Quadrate (Vorlage: die Tabelle der schrittweisen Berechnung in Lektion 1.1). Tragen Sie hier die vier Ergebnisse ein.

In [ ]:
# ZU ERGÄNZEN: Ihre von Hand berechneten Ergebnisse (Division durch n)
variance_tilleuls_by_hand = None   # in €²
variance_port_by_hand = None       # in €²
std_tilleuls_by_hand = None # in €
std_port_by_hand = None     # in €

In [ ]:
# Prüfung (Division durch n: ddof=0)
your_values = {
    'Varianz Tilleuls': (variance_tilleuls_by_hand, np.var(tilleuls, ddof=0)),
    'Varianz Port': (variance_port_by_hand, np.var(port, ddof=0)),
    'Standardabweichung Tilleuls': (std_tilleuls_by_hand, np.std(tilleuls, ddof=0)),
    'Standardabweichung Port': (std_port_by_hand, np.std(port, ddof=0)),
}
for name, (yours, expected) in your_values.items():
    if yours is None:
        print(f'{name}: noch nicht ausgefüllt.')
    else:
        print(f"{name}: {'richtig' if np.isclose(yours, expected) else 'bitte prüfen'}")

### d) Interpretieren

*Ihre Antwort:* …

### e) Die Standardabweichung programmieren

Vervollständigen Sie die Funktion nach Gleichung 1.3 des Buchs, mit einer Division durch $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Verwenden Sie weder `np.std` noch `np.var`: Ziel ist, jeden Schritt selbst zu schreiben.

In [ ]:
# Aufgabe 1.2 e) — ZU ERGÄNZEN
def std_dev(values):
    """Standardabweichung, Division durch n (Gleichung 1.3)."""
    x = np.array(values, dtype=float)
    n = len(x)
    mean = None   # ZU ERGÄNZEN: Gleichung 1.1
    deviations = None    # ZU ERGÄNZEN: eine Abweichung pro Wert
    variance = None  # ZU ERGÄNZEN: Mittelwert der Quadrate
    return None      # ZU ERGÄNZEN: zurück zur ursprünglichen Einheit

<details><summary><b>Hinweis</b> (klicken Sie, wenn Sie nicht weiterkommen)</summary>

Mit NumPy wirkt eine auf ein Array geschriebene Operation auf jeden Wert: `x - mean` liefert auf einmal die sechs Abweichungen und `deviations ** 2` ihre sechs Quadrate. `np.sum(...)` addiert, `np.sqrt(...)` zieht die Quadratwurzel.

</details>

In [ ]:
# Prüfung: Führen Sie diese Zelle aus, ohne sie zu ändern
if std_dev(tilleuls) is None:
    print('Die Funktion gibt noch None zurück: Bitte vervollständigen Sie sie.')
else:
    for name, one_series, expected in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtained = std_dev(one_series)
        status = 'richtig' if np.isclose(obtained, expected) else f'bitte prüfen (erwartet: {expected})'
        print(f'{name}: {obtained:.2f} € → {status}')

### f) Vertiefung: durch $n$ oder durch $n - 1$ teilen

Führen Sie die folgende Zelle aus. Finden Sie 10,95 von Hand, indem Sie die Summe der Quadrate durch $n - 1 = 5$ teilen. Welche pandas-Option liefert die Konvention des Kapitels?

*Ihre Antwort:* …

In [ ]:
rents = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, Standard (Division durch n − 1):', rents.std().round(2).to_dict())
print('pandas, ddof=0 (Division durch n)      :', rents.std(ddof=0).round(2).to_dict())

## Aufgabe 1.3 — Ein überraschender Verkauf?

*Daten D1.1. Begriffe: standardisierter Wert (Lektion 1.4, Gleichung 1.6).* Eine T2-Wohnung (zwei Zimmer) wurde gerade für 196 k€ verkauft. Mittelwerte und Standardabweichungen: die aus Lektion 1.1, mit Division durch $n$.

In [ ]:
# ZU ERGÄNZEN: Ihre auf Papier durchgeführten Berechnungen
z_north = None
z_south = None
south_price_as_unusual = None   # in k€, Frage c)

In [ ]:
# Prüfung (Standardabweichung mit Division durch n)
expected_values = {
    'z Nord': (z_north, (196 - np.mean(north)) / np.std(north, ddof=0)),
    'z Süd': (z_south, (196 - np.mean(south)) / np.std(south, ddof=0)),
    'gleichwertiger Preis Süd': (south_price_as_unusual,
                            np.mean(south) + (196 - np.mean(north)) / np.std(north, ddof=0) * np.std(south, ddof=0)),
}
for name, (yours, truth) in expected_values.items():
    status = 'noch nicht ausgefüllt' if yours is None else ('richtig' if np.isclose(yours, truth) else 'bitte prüfen')
    print(f'{name}: {status}')

*Ihre Antwort zu b):* …

## Aufgabe 1.4 — Gehören Fläche und Preis zusammen?

*Daten D0.1 (die sechs Wohnungen der Einleitung). Begriffe: Kovarianz und Korrelation (Lektion 1.5, Gleichungen 1.7 und 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(homes['floor_area'], homes['price'], color=BLUE)
for _, l in homes.iterrows():
    plt.annotate(l['home'], (l['floor_area'], l['price']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Wohnfläche (m²)')
plt.ylabel('Verkaufspreis (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Vor jeder Rechnung: Wird $r$ positiv oder negativ sein? Nahe bei 0, bei 0,5 oder bei 1?

*Ihre Antwort:* …

**b)** Vervollständigen Sie die folgenden Zeilen. Teilen Sie überall durch $n$.

In [ ]:
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
n = len(x)

products = None     # ZU ERGÄNZEN: (x_i − x̄)(y_i − ȳ) für jede Wohnung
covariance = None   # ZU ERGÄNZEN: Gleichung 1.7, Division durch n
r = None            # ZU ERGÄNZEN: Gleichung 1.8 (Standardabweichungen mit ddof=0)

<details><summary><b>Hinweis</b></summary>

`x.mean()` liefert den Mittelwert, `x - x.mean()` die sechs Abweichungen. Das Produkt `*` zweier Arrays multipliziert elementweise. `x.std(ddof=0)` liefert die Standardabweichung mit Division durch $n$.

</details>

In [ ]:
# Prüfung
if r is None or covariance is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Kovarianz:', round(covariance, 2), 'm²·k€ →',
          'richtig' if np.isclose(covariance, 1637.5) else 'bitte prüfen (erwartet: 1637,5)')
    print('r:', round(r, 4), '→', 'richtig' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'bitte prüfen')

**c)** Eine Korrelation nahe 1, aber Abweichungen von 5 bis 11 k€ im Betrag: Ist das ein Widerspruch?

*Ihre Antwort:* …

In [ ]:
print('np.cov standardmäßig (Division durch n − 1):', np.cov(x, y)[0, 1])
print('np.cov mit bias=True (Division durch n)     :', np.cov(x, y, bias=True)[0, 1])

## Aufgabe 1.5 — Vorhersagen, dann prüfen

*Daten D1.1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1), Median und Interquartilsabstand (Lektion 1.3).* Eine sechste T2-Wohnung wird im Norden für 240 k€ verkauft.

**a)** Notieren Sie vor jeder Rechnung Ihre Vorhersagen: Steigen Mittelwert, Median, Standardabweichung und Interquartilsabstand stark, ein wenig oder gar nicht?

*Ihre Vorhersagen:* …

In [ ]:
north_plus = north + [240]
print('Nord:', sorted(north))
print('Nord mit dem Verkauf zu 240 k€:', sorted(north_plus))

**b)** Vervollständigen Sie die Funktion `summary`. Sie soll die vier Kennzahlen zurückgeben, mit der Standardabweichung durch $n$ geteilt und den Quartilen nach der Regel des Buchs berechnet (`book_quartiles`, weiter oben bereitgestellt). Die Schlüssel des Wörterbuchs sind englisch: `mean` = Mittelwert, `median` = Median, `std_dev` = Standardabweichung, `iqr` = Interquartilsabstand.

In [ ]:
# ZU ERGÄNZEN
def summary(one_series):
    q1, q3 = book_quartiles(one_series)
    return {
        'mean': None,              # ZU ERGÄNZEN
        'median': None,              # ZU ERGÄNZEN: np.median
        'std_dev': None,           # ZU ERGÄNZEN: Division durch n
        'iqr': None,  # ZU ERGÄNZEN: aus q1 und q3
    }

In [ ]:
table = pd.DataFrame({'Nord (5 Verk.)': summary(north), 'Nord + 240 k€ (6 Verk.)': summary(north_plus),
                        'Süd (5 Verk.)': summary(south)})
if table.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion summary.')
else:
    expected = {'mean': (180, 190), 'median': (181, 182), 'std_dev': (4, 22.66),
               'iqr': (6, 8)}
    for k, (before, after) in expected.items():
        ok = np.isclose(table.loc[k].iloc[0], before) and np.isclose(table.loc[k].iloc[1], after, atol=0.005)
        print(f"{k}: {'richtig' if ok else 'bitte prüfen'}")
table.round(2)

In [ ]:
# Die beiden Reihen nach dem Vorbild der Abbildung „Mittelwert ± Standardabweichung“ aus Lektion 1.1
if table.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion summary, um das Diagramm anzuzeigen.')
else:
    dot_rows({'Nord': north, 'Nord + 240': north_plus},
                      'Verkaufspreis einer T2-Wohnung (k€)',
                      bands={'Nord': (np.mean(north), np.std(north, ddof=0)),
                              'Nord + 240': (np.mean(north_plus), np.std(north_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** Sind die Preise im Norden unvorhersehbarer geworden als die im Süden?

*Ihre Antwort:* …

## Aufgabe 1.6 — Herausforderung: zwei Geraden, derselbe mittlere Fehler

*Daten D0.1. Begriffe: vorzeichenbehaftete Abweichung und Betrag (Einleitung, Lektion 1.1), Standardabweichung (Lektion 1.1).* Nach Augenmaß gezeichnete Gerade: vorhergesagter Preis = 3 × Fläche + 30. Faule Gerade: vorhergesagter Preis = 222 k€ für alle Wohnungen.

In [ ]:
price = homes['price'].to_numpy(dtype=float)
floor_area = homes['floor_area'].to_numpy(dtype=float)

eye_line_errors = None          # ZU ERGÄNZEN: beobachtet − vorhergesagt, nach Augenmaß gezeichnete Gerade
lazy_line_errors = None    # ZU ERGÄNZEN: beobachtet − vorhergesagt, faule Gerade

In [ ]:
# Die drei Kennzahlen aus Frage b) (Standardabweichung: Division durch n)
if eye_line_errors is None or lazy_line_errors is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    indicators = pd.DataFrame({
        name: {'Mittel vorzeichenbeh. Abweichungen': e.mean(),
              'Mittel der Abweichungsbeträge': np.abs(e).mean(),
              'Std.-abw. vorzeichenbeh. Abweich.': e.std(ddof=0)}
        for name, e in [('Gerade nach Augenmaß', eye_line_errors), ('faule Gerade', lazy_line_errors)]
    })
    print('Abweichungen, Gerade nach Augenmaß:', eye_line_errors)
    print('Abweichungen, faule Gerade        :', lazy_line_errors)
    print('Prüfung:', 'richtig' if np.allclose(indicators.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicators.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'bitte prüfen')
    display(indicators.round(2))

In [ ]:
if eye_line_errors is not None and lazy_line_errors is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(floor_area - 0.8, eye_line_errors, color=ORANGE, label="Gerade nach Augenmaß")
    ax.scatter(floor_area + 0.8, lazy_line_errors, color=BLUE, marker='s', label='faule Gerade')
    ax.set_xlabel('Wohnfläche (m²)')
    ax.set_ylabel('Abweichung beobachtet − vorhergesagt (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Ihre Antworten zu c) und d):* …

In [ ]:
# Kontrolle zu d): Die Standardabweichung der Abweichungen der faulen Geraden ist gleich der der Preise
if lazy_line_errors is not None:
    print(round(lazy_line_errors.std(ddof=0), 4), round(price.std(ddof=0), 4))